# SPEC 008 — Package the model and measure the inference settings (Kaggle GPU)

Packages the trained model with checksums, then measures three inference settings on validation
cases. **The test set is not touched** (D-014).

**Before running**
1. *Add Input*: the outputs of `kits23-stage-1..3`, `kits23-prepare-data` and `kits23-train-2`.
2. *Settings*: **Accelerator = GPU T4 x1**, **Internet = On**.
3. **Save Version → Save & Run All (Commit)**. Expected GPU time: about 40 minutes.

The code cells are generated by `scripts/notebooks/build_release_notebook.py`.
Edit the repository, not this notebook.

In [ ]:
# Step 0 - machine check and folders
import json, os, shutil, subprocess, sys, time
from pathlib import Path
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])
WORK = "/kaggle/working/project23d"
for sub in ['reports/release', 'scripts/data', 'scripts/eval', 'scripts/inference', 'scripts/release', 'src/project3d', 'src/project3d/data', 'src/project3d/eval', 'src/project3d/inference', 'src/project3d/training']:
    os.makedirs(f"{WORK}/{sub}", exist_ok=True)
T0 = time.time()

Write the project code (generated from the repository).

In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/__init__.py
"""Project_23D: renal CT 3D segmentation."""


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/__init__.py
"""KiTS23 data pipeline (SPEC 002): verify, features, split, nnU-Net export."""


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/manifest.py
"""Locate KiTS23 files and verify them against the pinned checksum manifest.

The manifest is ``data/sources/kits23_v0.1.4_files.tsv`` (D-011): one row per
file with ``case_id, kind, size_bytes, sha256, url``. ``kind`` is ``image`` or
``segmentation``.
"""

from __future__ import annotations

import csv
import hashlib
import re
from dataclasses import dataclass
from pathlib import Path

KINDS = ("image", "segmentation")
CASE_FILE_RE = re.compile(r"^(case_\d{5})\.nii(\.gz)?$")

# Verification statuses. Only OK is a pass; DECOMPRESSED is a warning (the file
# exists but the platform un-gzipped it, so the published hash cannot apply).
OK = "OK"
MISSING = "MISSING"
SIZE_MISMATCH = "SIZE_MISMATCH"
HASH_MISMATCH = "HASH_MISMATCH"
DECOMPRESSED = "DECOMPRESSED_UNVERIFIED"
DUPLICATE = "DUPLICATE"
FAILING = {MISSING, SIZE_MISMATCH, HASH_MISMATCH, DUPLICATE}


@dataclass(frozen=True)
class Expected:
    case_id: str
    kind: str
    size_bytes: int
    sha256: str


def load_manifest(tsv_path: str | Path) -> dict[tuple[str, str], Expected]:
    """Read the checksum manifest, keyed by ``(case_id, kind)``."""
    out: dict[tuple[str, str], Expected] = {}
    with open(tsv_path, newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh, delimiter="\t"):
            if row["kind"] not in KINDS:
                raise ValueError(f"unknown kind {row['kind']!r} for {row['case_id']}")
            key = (row["case_id"], row["kind"])
            if key in out:
                raise ValueError(f"duplicate manifest row {key}")
            out[key] = Expected(row["case_id"], row["kind"], int(row["size_bytes"]), row["sha256"].lower())
    return out


def find_case_files(root: str | Path) -> dict[str, list[Path]]:
    """Recursively find ``case_XXXXX.nii.gz`` (or ``.nii``) files under ``root``.

    Returns every path per case so duplicates can be reported, not silently picked.
    """
    found: dict[str, list[Path]] = {}
    for path in sorted(Path(root).rglob("case_*")):
        m = CASE_FILE_RE.match(path.name)
        if m and path.is_file():
            found.setdefault(m.group(1), []).append(path)
    return found


def sha256_file(path: str | Path, chunk: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while block := fh.read(chunk):
            h.update(block)
    return h.hexdigest()


def verify_kind(
    manifest: dict[tuple[str, str], Expected],
    kind: str,
    root: str | Path,
    check_hash: bool = True,
) -> list[dict]:
    """Check every manifest entry of one kind against the files under ``root``.

    Returns one report row per manifest entry, plus one row per file found
    under ``root`` that the manifest does not list (status ``UNEXPECTED``).
    """
    found = find_case_files(root)
    rows = []
    for (case_id, k), exp in sorted(manifest.items()):
        if k != kind:
            continue
        paths = found.pop(case_id, [])
        row = {"case_id": case_id, "kind": kind, "path": "", "status": OK,
               "size_bytes": "", "sha256": ""}
        if not paths:
            row["status"] = MISSING
        elif len(paths) > 1:
            row.update(status=DUPLICATE, path=";".join(str(p) for p in paths))
        else:
            path = paths[0]
            row["path"] = str(path)
            row["size_bytes"] = path.stat().st_size
            if path.suffix != ".gz":
                row["status"] = DECOMPRESSED
            elif row["size_bytes"] != exp.size_bytes:
                row["status"] = SIZE_MISMATCH
            elif check_hash:
                row["sha256"] = sha256_file(path)
                if row["sha256"] != exp.sha256:
                    row["status"] = HASH_MISMATCH
        rows.append(row)
    for case_id, paths in sorted(found.items()):
        for p in paths:
            rows.append({"case_id": case_id, "kind": kind, "path": str(p),
                         "status": "UNEXPECTED", "size_bytes": p.stat().st_size, "sha256": ""})
    return rows


def summarize(rows: list[dict]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for r in rows:
        counts[r["status"]] = counts.get(r["status"], 0) + 1
    return counts


def resolved_paths(rows: list[dict]) -> dict[str, Path]:
    """Map case_id to its single verified path (OK or DECOMPRESSED rows only)."""
    return {r["case_id"]: Path(r["path"]) for r in rows if r["status"] in (OK, DECOMPRESSED)}


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/features.py
"""Per-case label features used to stratify the split (D-012).

Only segmentations are read. Labels: 0 background, 1 kidney, 2 tumor, 3 cyst.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np

VALID_LABELS = {0, 1, 2, 3}
KITS21_LAST_ID = 299  # cases 0-299 are the KiTS21 cohort; 400+ were added in KiTS23


def cohort_of(case_id: str) -> str:
    """Cohort proxy for contrast phase (D-010, D-012); not a phase label."""
    return "kits21" if int(case_id.split("_")[1]) <= KITS21_LAST_ID else "kits23_new"


def label_features(seg: np.ndarray, spacing_mm: tuple[float, ...]) -> dict:
    """Volumes in millilitres and presence flags from a label array."""
    present = {int(v) for v in np.unique(seg)}
    unknown = present - VALID_LABELS
    if unknown:
        raise ValueError(f"unexpected label values {sorted(unknown)}")
    counts = np.bincount(seg.ravel().astype(np.int64), minlength=4)
    voxel_ml = float(np.prod(spacing_mm)) / 1000.0
    return {
        "kidney_ml": round(counts[1] * voxel_ml, 3),
        "tumor_ml": round(counts[2] * voxel_ml, 3),
        "cyst_ml": round(counts[3] * voxel_ml, 3),
        "has_tumor": bool(counts[2] > 0),
        "has_cyst": bool(counts[3] > 0),
    }


def case_features(case_id: str, seg_path: str | Path) -> dict:
    """Read one segmentation and return its features plus header geometry."""
    import nibabel as nib

    img = nib.load(str(seg_path))
    seg = np.asanyarray(img.dataobj)
    if not np.all(np.mod(seg, 1) == 0):
        raise ValueError(f"{case_id}: non-integer label values")
    seg = seg.astype(np.uint8)
    spacing = tuple(float(z) for z in img.header.get_zooms()[:3])
    row = {"case_id": case_id, "cohort": cohort_of(case_id),
           "shape": "x".join(map(str, seg.shape)),
           "spacing_mm": "x".join(f"{s:.4f}" for s in spacing)}
    row.update(label_features(seg, spacing))
    return row


def geometry_mismatch(image_path: str | Path, seg_path: str | Path, atol: float = 1e-3) -> str | None:
    """Compare image and mask headers (no voxel data read). None if they match."""
    import nibabel as nib

    img, seg = nib.load(str(image_path)), nib.load(str(seg_path))
    if img.shape[:3] != seg.shape[:3]:
        return f"shape {img.shape[:3]} vs {seg.shape[:3]}"
    if not np.allclose(img.affine, seg.affine, atol=atol):
        return "affine differs"
    return None


def add_size_bins(rows: list[dict]) -> tuple[float, float]:
    """Add ``size_bin`` in place: tertiles of tumor volume among tumor cases.

    Cases without tumor get ``none``. Returns the two cut points (ml) so they
    can be recorded with the split.
    """
    vols = np.array([r["tumor_ml"] for r in rows if r["has_tumor"]], dtype=float)
    if vols.size == 0:
        cuts = (0.0, 0.0)
    else:
        q1, q2 = np.quantile(vols, [1 / 3, 2 / 3])
        cuts = (float(q1), float(q2))
    for r in rows:
        if not r["has_tumor"]:
            r["size_bin"] = "none"
        elif r["tumor_ml"] <= cuts[0]:
            r["size_bin"] = "small"
        elif r["tumor_ml"] <= cuts[1]:
            r["size_bin"] = "medium"
        else:
            r["size_bin"] = "large"
    return cuts


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/split.py
"""Case-level stratified train/validation/test split (SPEC 002, D-012).

Strata are ``cohort | size_bin | cyst``. Per-stratum counts are allocated by
largest-remainder apportionment, so the global counts hit the target fractions
exactly and every stratum is represented proportionally. Cases are shuffled
within each stratum with a fixed seed.

The saved ``split.json`` is the authority. Its fingerprint is recorded so any
later change is detectable; the split is never regenerated after training.
"""

from __future__ import annotations

import hashlib
import json
import math
import random

ROLES = ("train", "val", "test")


def stratum_key(row: dict) -> str:
    return f"{row['cohort']}|{row['size_bin']}|{'cyst' if row['has_cyst'] else 'nocyst'}"


def _apportion(sizes: dict[str, int], total: int, capacity: dict[str, int]) -> dict[str, int]:
    """Split ``total`` across strata proportionally to ``sizes``, within ``capacity``."""
    n = sum(sizes.values())
    if n == 0 or total == 0:
        return {k: 0 for k in sizes}
    quota = {k: sizes[k] * total / n for k in sizes}
    alloc = {k: min(math.floor(quota[k]), capacity[k]) for k in sizes}
    remaining = total - sum(alloc.values())
    # Largest remainder first; ties broken by stratum name for determinism.
    order = sorted(sizes, key=lambda k: (-(quota[k] - math.floor(quota[k])), k))
    while remaining > 0:
        progressed = False
        for k in order:
            if remaining == 0:
                break
            if alloc[k] < capacity[k]:
                alloc[k] += 1
                remaining -= 1
                progressed = True
        if not progressed:
            raise ValueError("not enough cases to satisfy the requested split")
    return alloc


def make_split(rows: list[dict], seed: int = 42,
               fractions: tuple[float, float, float] = (0.70, 0.15, 0.15)) -> dict[str, str]:
    """Return ``{case_id: role}`` with role in ``train``/``val``/``test``."""
    if abs(sum(fractions) - 1.0) > 1e-9:
        raise ValueError("fractions must sum to 1")
    ids = [r["case_id"] for r in rows]
    if len(set(ids)) != len(ids):
        raise ValueError("duplicate case_id in feature rows")

    strata: dict[str, list[str]] = {}
    for r in rows:
        strata.setdefault(stratum_key(r), []).append(r["case_id"])
    sizes = {k: len(v) for k, v in strata.items()}
    n = len(rows)
    n_test = round(n * fractions[2])
    n_val = round(n * fractions[1])

    test_alloc = _apportion(sizes, n_test, sizes)
    val_alloc = _apportion(sizes, n_val, {k: sizes[k] - test_alloc[k] for k in sizes})

    rng = random.Random(seed)
    roles: dict[str, str] = {}
    for key in sorted(strata):
        members = sorted(strata[key])
        rng.shuffle(members)
        t, v = test_alloc[key], val_alloc[key]
        for i, cid in enumerate(members):
            roles[cid] = "test" if i < t else "val" if i < t + v else "train"
    return dict(sorted(roles.items()))


def fingerprint(roles: dict[str, str]) -> str:
    """SHA-256 of the canonical ``{case_id: role}`` mapping."""
    canon = json.dumps(dict(sorted(roles.items())), sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canon.encode("utf-8")).hexdigest()


def nnunet_splits(roles: dict[str, str]) -> list[dict[str, list[str]]]:
    """nnU-Net ``splits_final.json`` content: one fold, train vs val. Test is excluded."""
    return [{
        "train": sorted(c for c, r in roles.items() if r == "train"),
        "val": sorted(c for c, r in roles.items() if r == "val"),
    }]


def summary(rows: list[dict], roles: dict[str, str]) -> dict:
    """Counts per role, overall and by each stratification variable."""
    out: dict = {"total": {r: 0 for r in ROLES}}
    for var in ("cohort", "size_bin", "has_cyst"):
        out[var] = {}
    for row in rows:
        role = roles[row["case_id"]]
        out["total"][role] += 1
        for var in ("cohort", "size_bin", "has_cyst"):
            bucket = out[var].setdefault(str(row[var]), {r: 0 for r in ROLES})
            bucket[role] += 1
    return out


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/data/nnunet_export.py
"""Build the nnU-Net v2 raw dataset folder for KiTS23 (SPEC 002).

Mirrors nnU-Net's own ``dataset_conversion/Dataset220_KiTS2023.py``: region-based
labels (kidney+masses, masses, tumor), ``regions_class_order`` (1, 3, 2) and the
``NibabelIOWithReorient`` reader. Two differences, both deliberate:

- only train and validation cases go into ``imagesTr``/``labelsTr``, so the
  locked test cases cannot influence nnU-Net's planning or normalization;
- files are symlinked by default instead of copied, so no extra disk is used.
"""

from __future__ import annotations

import json
import os
import shutil
from pathlib import Path

DATASET_ID = 220
DATASET_NAME = "KiTS2023"
DATASET_JSON = {
    "channel_names": {"0": "CT"},
    "labels": {"background": 0, "kidney": [1, 2, 3], "masses": [2, 3], "tumor": 2},
    "regions_class_order": [1, 3, 2],
    "overwrite_image_reader_writer": "NibabelIOWithReorient",
    "name": DATASET_NAME,
    "release": "0.1.4",
    "reference": "https://github.com/neheller/kits23",
    "licence": "CC BY-NC-SA 4.0 (KiTS23 data terms)",
    "description": "KiTS23 train+val cases only; locked test cases excluded (D-012)",
}


def dataset_folder_name(dataset_id: int = DATASET_ID, name: str = DATASET_NAME) -> str:
    return f"Dataset{dataset_id:03d}_{name}"


def _file_ending(paths: list[Path]) -> str:
    endings = {".nii.gz" if p.name.endswith(".nii.gz") else p.suffix for p in paths}
    if len(endings) != 1:
        raise ValueError(f"mixed file endings {sorted(endings)}; nnU-Net needs one")
    return endings.pop()


def _place(src: Path, dst: Path, mode: str) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    if mode == "symlink":
        os.symlink(src.resolve(), dst)
    elif mode == "copy":
        shutil.copy2(src, dst)
    else:
        raise ValueError(f"unknown mode {mode!r}")


def build_raw_dataset(roles: dict[str, str], images: dict[str, Path], segs: dict[str, Path],
                      raw_root: str | Path, mode: str = "symlink",
                      dataset_id: int = DATASET_ID) -> Path:
    """Create ``<raw_root>/DatasetXXX_KiTS2023`` and return its path.

    ``imagesTr``/``labelsTr`` hold train+val cases; ``imagesTs`` holds test images.
    Test labels are never placed in the nnU-Net folder.
    """
    trainval = sorted(c for c, r in roles.items() if r in ("train", "val"))
    test = sorted(c for c, r in roles.items() if r == "test")
    missing = [c for c in roles if c not in images or (c in trainval and c not in segs)]
    if missing:
        raise ValueError(f"no verified file for {len(missing)} cases, e.g. {missing[:3]}")

    ending = _file_ending([images[c] for c in roles] + [segs[c] for c in trainval])
    out = Path(raw_root) / dataset_folder_name(dataset_id)
    if out.exists():
        shutil.rmtree(out)
    for c in trainval:
        _place(images[c], out / "imagesTr" / f"{c}_0000{ending}", mode)
        _place(segs[c], out / "labelsTr" / f"{c}{ending}", mode)
    for c in test:
        _place(images[c], out / "imagesTs" / f"{c}_0000{ending}", mode)

    meta = dict(DATASET_JSON, numTraining=len(trainval), file_ending=ending)
    (out / "dataset.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return out


def check_no_test_leak(dataset_dir: str | Path, roles: dict[str, str]) -> None:
    """Fail if any test case appears in imagesTr or labelsTr."""
    test = {c for c, r in roles.items() if r == "test"}
    for sub in ("imagesTr", "labelsTr"):
        for p in (Path(dataset_dir) / sub).iterdir():
            if p.name[:10] in test:
                raise AssertionError(f"test case {p.name[:10]} found in {sub}")


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/eval/__init__.py
"""SPEC 004-005: test-set metrics, statistics and case analysis."""


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/eval/metrics.py
"""Secondary per-case metrics (SPEC 004).

The co-primary Dice and surface Dice come from the official KiTS23 evaluator
(see ``official.py``). This module adds the secondary metrics from
docs/evaluation.md, with the official empty-mask rule for Dice and IoU:
both empty -> 1, exactly one empty -> 0. Precision, recall and HD95 are
undefined (NaN) when their denominator mask is empty; counts are reported.
"""

from __future__ import annotations

import numpy as np
from scipy import ndimage

# Official KiTS hierarchical evaluation classes first, then the remaining single classes.
HECS = {"kidney_and_mass": (1, 2, 3), "mass": (2, 3), "tumor": (2,)}
REGIONS = {**HECS, "kidney": (1,), "cyst": (3,)}
# Copied from kits23.configuration.labels; official.check_config() asserts they still match.
SD_TOLERANCES_MM = {"kidney_and_mass": 1.0330772532390826, "mass": 1.1328796488598762,
                    "tumor": 1.1498198361434828}


def region_mask(seg: np.ndarray, labels: tuple[int, ...]) -> np.ndarray:
    return np.isin(seg, labels)


def overlap(pred: np.ndarray, ref: np.ndarray) -> dict:
    """Dice, IoU, precision and recall for two boolean masks."""
    tp = int(np.count_nonzero(pred & ref))
    n_pred, n_ref = int(np.count_nonzero(pred)), int(np.count_nonzero(ref))
    if n_pred == 0 and n_ref == 0:
        dice = iou = 1.0
    elif n_pred == 0 or n_ref == 0:
        dice = iou = 0.0
    else:
        dice = 2 * tp / (n_pred + n_ref)
        iou = tp / (n_pred + n_ref - tp)
    return {"dice": dice, "iou": iou,
            "precision": tp / n_pred if n_pred else float("nan"),
            "recall": tp / n_ref if n_ref else float("nan")}


def _surface(mask: np.ndarray) -> np.ndarray:
    return mask & ~ndimage.binary_erosion(mask, border_value=0)


def hd95(pred: np.ndarray, ref: np.ndarray, spacing_mm: tuple[float, ...]) -> float:
    """95th-percentile symmetric Hausdorff distance in mm (max of the two directions).

    NaN when either mask is empty. Computed inside the bounding box of both
    masks, which holds every surface voxel, so distances are unchanged.
    """
    if not pred.any() or not ref.any():
        return float("nan")
    idx = np.argwhere(pred | ref)
    lo, hi = np.maximum(idx.min(0) - 1, 0), idx.max(0) + 2
    box = tuple(slice(a, b) for a, b in zip(lo, hi, strict=True))
    p = np.pad(pred[box], 1)
    r = np.pad(ref[box], 1)
    sp, sr = _surface(p), _surface(r)
    dist_to_r = ndimage.distance_transform_edt(~sr, sampling=spacing_mm)
    dist_to_p = ndimage.distance_transform_edt(~sp, sampling=spacing_mm)
    return float(max(np.percentile(dist_to_r[sp], 95), np.percentile(dist_to_p[sr], 95)))


def case_metrics(pred_seg: np.ndarray, ref_seg: np.ndarray, spacing_mm: tuple[float, ...]) -> dict:
    """All secondary metrics for one case, flattened as ``<region>_<metric>``."""
    if pred_seg.shape != ref_seg.shape:
        raise ValueError(f"shape {pred_seg.shape} vs {ref_seg.shape}")
    voxel_ml = float(np.prod(spacing_mm)) / 1000.0
    out: dict = {}
    for name, labels in REGIONS.items():
        p, r = region_mask(pred_seg, labels), region_mask(ref_seg, labels)
        for k, v in overlap(p, r).items():
            out[f"{name}_{k}"] = v
        out[f"{name}_hd95_mm"] = hd95(p, r, spacing_mm)
        out[f"{name}_ref_ml"] = round(float(np.count_nonzero(r)) * voxel_ml, 3)
        out[f"{name}_pred_ml"] = round(float(np.count_nonzero(p)) * voxel_ml, 3)
    return out


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/eval/stats.py
"""Per-case distribution summaries and bootstrap confidence intervals."""

from __future__ import annotations

import numpy as np

N_BOOT = 10_000
MIN_CASES_FOR_CI = 10  # below this a CI is not meaningful; reported as None


def describe(values) -> dict:
    """n, mean, median, IQR, 10th percentile, min and max, ignoring NaN (counted)."""
    a = np.asarray(values, dtype=float)
    ok = a[~np.isnan(a)]
    out = {"n": int(ok.size), "n_undefined": int(a.size - ok.size)}
    if ok.size == 0:
        return out
    q10, q1, med, q3 = np.percentile(ok, [10, 25, 50, 75])
    out.update(mean=float(ok.mean()), median=float(med), q1=float(q1), q3=float(q3),
               p10=float(q10), min=float(ok.min()), max=float(ok.max()))
    return out


def bootstrap_ci(values, n_boot: int = N_BOOT, seed: int = 42, alpha: float = 0.05):
    """Percentile bootstrap CI of the mean over cases; None if too few cases."""
    a = np.asarray(values, dtype=float)
    a = a[~np.isnan(a)]
    if a.size < MIN_CASES_FOR_CI:
        return None
    rng = np.random.default_rng(seed)
    means = a[rng.integers(0, a.size, size=(n_boot, a.size))].mean(axis=1)
    lo, hi = np.percentile(means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return [float(lo), float(hi)]


def describe_with_ci(values, **kw) -> dict:
    return {**describe(values), "ci95": bootstrap_ci(values, **kw)}


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/eval/official.py
"""Thin wrapper around the official KiTS23 evaluator (co-primary metrics, D-012).

Requires ``pip install git+https://github.com/neheller/kits23`` (done in the
Kaggle notebook). The official ``compute_metrics_for_case`` returns a 3x2
array: HECs (kidney_and_mass, mass, tumor) x (Dice, surface Dice). Its own
shape/spacing checks are ineffective, so evaluate.py checks geometry first.
"""

from __future__ import annotations

from .metrics import HECS, SD_TOLERANCES_MM


def check_config() -> None:
    """Fail if the installed evaluator's classes or tolerances differ from ours."""
    from kits23.configuration import labels

    assert list(labels.HEC_NAME_LIST) == list(HECS), labels.HEC_NAME_LIST
    for name, lab in HECS.items():
        assert tuple(labels.KITS_HEC_LABEL_MAPPING[name]) == lab, name
        assert abs(labels.HEC_SD_TOLERANCES_MM[name] - SD_TOLERANCES_MM[name]) < 1e-12, name


def official_case(pred_path: str, ref_path: str) -> dict:
    from kits23.evaluation.metrics import compute_metrics_for_case

    m = compute_metrics_for_case(str(pred_path), str(ref_path))
    out = {}
    for i, name in enumerate(HECS):
        out[f"{name}_dice_official"] = float(m[i, 0])
        out[f"{name}_sd_official"] = float(m[i, 1])
    return out


def official_version() -> str:
    """Installed kits23 version plus the git commit pip recorded, if any."""
    import importlib.metadata as md
    import json

    dist = md.distribution("kits23")
    commit = ""
    direct = dist.read_text("direct_url.json")
    if direct:
        commit = json.loads(direct).get("vcs_info", {}).get("commit_id", "")
    return f"{dist.version} {commit}".strip()


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/eval/postprocess.py
"""Post-processing chosen on the fold-0 validation cases (D-021)."""

from __future__ import annotations

import numpy as np
from scipy import ndimage

N_KIDNEYS = 2
CONNECTIVITY = np.ones((3, 3, 3), dtype=bool)  # 26-connected


def keep_largest_regions(seg: np.ndarray, n: int = N_KIDNEYS) -> np.ndarray:
    """Clear everything except the ``n`` largest connected foreground regions.

    Labels inside the kept regions do not change. A label map with ``n`` regions or
    fewer is returned as is.
    """
    regions, count = ndimage.label(seg > 0, structure=CONNECTIVITY)
    if count <= n:
        return seg
    sizes = np.bincount(regions.ravel())
    sizes[0] = 0  # background
    keep = np.argsort(sizes)[-n:]
    return np.where(np.isin(regions, keep), seg, 0).astype(seg.dtype)


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/inference/__init__.py
"""Inference used by the command line, the benchmark and the live service."""


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/inference/predictor.py
"""SPEC 008: the predictor used by the command line, the benchmark and the live service.

``Predictor`` holds everything that must stay true for every caller: input checks, the
post-processing rule from evaluation (D-021), the geometry check and the region volumes.
``NnUNetModel`` is the only part that needs nnU-Net and a GPU, so the rest is tested locally
with a fake model.
"""

from __future__ import annotations

import shutil
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Protocol

import numpy as np

from project3d.data import features
from project3d.eval.postprocess import keep_largest_regions

CHECKPOINT = "checkpoint_final.pth"
TRAINER_FILE = "project23d_trainer.py"  # the name it gets inside nnU-Net's package
LABELS = {"kidney": 1, "tumor": 2, "cyst": 3}
MAX_VOXELS = 300_000_000  # about 512 x 512 x 1150, larger than any scan used in training
SPACING_RANGE_MM = (0.1, 20.0)


class InvalidInput(ValueError):
    """The scan can't be used. The message is written for the person who supplied it."""


@dataclass(frozen=True)
class Options:
    """Inference settings. The defaults are the setting used in the evaluation (SPEC 004)."""

    tta: bool = True
    tile_step: float = 0.5
    device: str = "cuda"

    def __post_init__(self) -> None:
        if not 0 < self.tile_step <= 1:
            raise ValueError(f"tile_step must be in (0, 1], got {self.tile_step}")


class Model(Protocol):
    def read(self, path: Path) -> tuple[np.ndarray, dict]: ...
    def predict(self, data: np.ndarray, properties: dict) -> np.ndarray: ...
    def write(self, seg: np.ndarray, path: Path, properties: dict) -> None: ...


def check_header(shape: tuple, spacing: tuple, max_voxels: int = MAX_VOXELS) -> None:
    """Raise InvalidInput for a scan that isn't 3D, has bad spacing or is too large."""
    if len(shape) != 3:
        raise InvalidInput(f"Expected a 3D scan, but this file has {len(shape)} dimensions {tuple(shape)}.")
    low, high = SPACING_RANGE_MM
    if not all(np.isfinite(s) and low <= s <= high for s in spacing[:3]):
        raise InvalidInput(f"Voxel spacing {tuple(float(s) for s in spacing[:3])} mm is invalid: "
                           f"each axis must be between {low} and {high} mm.")
    voxels = int(np.prod([int(n) for n in shape]))
    if voxels > max_voxels:
        raise InvalidInput(f"This scan has {voxels:,} voxels; the limit is {max_voxels:,}.")


def load_checked(path: str | Path, max_voxels: int = MAX_VOXELS):
    """Open a NIfTI file and check its header. Voxel data is not read."""
    import nibabel as nib

    try:
        img = nib.load(str(path))
    except Exception as exc:
        raise InvalidInput("The file could not be read as a NIfTI scan.") from exc
    check_header(img.shape, img.header.get_zooms(), max_voxels)
    return img


def region_volumes(seg: np.ndarray, spacing: tuple) -> dict[str, float]:
    """Volume in mL of each region. One mL is 1000 cubic millimetres."""
    voxel_ml = float(np.prod([float(s) for s in spacing[:3]])) / 1000
    return {f"{name}_ml": round(float((seg == label).sum()) * voxel_ml, 3) for name, label in LABELS.items()}


class Predictor:
    def __init__(self, model: Model, options: Options | None = None):
        self.model = model
        self.options = options or Options()

    def predict_file(self, src: str | Path, dst: str | Path) -> dict[str, Any]:
        """Segment one scan, write the labels to ``dst`` and return a summary.

        The summary has no file names, so a caller can log or store it.
        """
        src, dst = Path(src), Path(dst)
        img = load_checked(src)
        data, properties = self.model.read(src)
        seg = keep_largest_regions(np.asarray(self.model.predict(data, properties)).astype(np.uint8))
        self.model.write(seg, dst, properties)
        why = features.geometry_mismatch(dst, src)
        if why:
            raise RuntimeError(f"output does not match the input geometry ({why})")
        spacing = tuple(float(z) for z in img.header.get_zooms()[:3])
        return {"shape": [int(n) for n in img.shape], "spacing_mm": list(spacing),
                "voxel_count": int(np.prod([int(n) for n in img.shape])),
                "volumes_ml": region_volumes(seg, spacing), "options": asdict(self.options),
                "postprocessing": "keep the two largest foreground regions"}


def install_trainer(trainer_file: Path, variants_dir: Path | None = None) -> Path:
    """Copy our trainer into nnU-Net, which only looks for trainer classes in its own package.

    The checkpoint names that class, so loading fails without this step.
    """
    if variants_dir is None:
        import nnunetv2

        variants_dir = Path(nnunetv2.__file__).parent / "training" / "nnUNetTrainer" / "variants"
    return Path(shutil.copy(trainer_file, variants_dir / TRAINER_FILE))


class NnUNetModel:
    """The packaged nnU-Net model, reading files the same way training did."""

    def __init__(self, model_dir: str | Path, options: Options | None = None):
        options = options or Options()
        model_dir = Path(model_dir)
        install_trainer(model_dir / TRAINER_FILE)
        import torch
        from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

        self.predictor = nnUNetPredictor(tile_step_size=options.tile_step, use_mirroring=options.tta,
                                         device=torch.device(options.device), allow_tqdm=False)
        self.predictor.initialize_from_trained_model_folder(str(model_dir), use_folds=(0,),
                                                            checkpoint_name=CHECKPOINT)
        self.io = self.predictor.plans_manager.image_reader_writer_class()

    def read(self, path: Path) -> tuple[np.ndarray, dict]:
        return self.io.read_images([str(path)])

    def predict(self, data: np.ndarray, properties: dict) -> np.ndarray:
        # runs in this process: nnU-Net's worker processes can't be used inside a ZeroGPU call
        return self.predictor.predict_single_npy_array(data, properties)

    def write(self, seg: np.ndarray, path: Path, properties: dict) -> None:
        self.io.write_seg(seg, str(path), properties)


In [ ]:
%%writefile /kaggle/working/project23d/src/project3d/training/nnunet_trainer.py
"""nnU-Net trainer used for SPEC 003 (D-016).

Copied into the installed ``nnunetv2/training/nnUNetTrainer/variants`` folder on
Kaggle, because nnU-Net finds trainers by class name inside its own package. It
imports nothing from this project.
"""

import os

import torch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer


class nnUNetTrainer_100epochs_kaggle(nnUNetTrainer):
    """``nnUNetTrainer_100epochs`` with a checkpoint every 5 epochs instead of 50.

    Same schedule, model and augmentation. A Kaggle session ends after 12 hours,
    so at most 5 epochs are lost when a session stops and the next one resumes.
    Final validation resumes too: cases already predicted in ``validation/`` are
    skipped, and the summary still scores every file in that folder.
    """

    def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict,
                 device: torch.device = torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.num_epochs = 100
        self.save_every = 5
        self.skip_predicted = False

    def do_split(self):
        tr_keys, val_keys = super().do_split()
        if self.skip_predicted:
            folder = os.path.join(self.output_folder, "validation")
            val_keys = [k for k in val_keys if not os.path.exists(os.path.join(folder, f"{k}.nii.gz"))]
            print(f"final validation: {len(val_keys)} cases left to predict", flush=True)
        return tr_keys, val_keys

    def perform_actual_validation(self, save_probabilities: bool = False):
        self.skip_predicted = True
        super().perform_actual_validation(save_probabilities)


In [ ]:
%%writefile /kaggle/working/project23d/scripts/data/prepare_kits23.py
"""SPEC 002 pipeline: verify -> features -> split -> nnU-Net raw dataset.

Runs on a Kaggle CPU session (or locally on a small subset). Example::

    python scripts/data/prepare_kits23.py --input-root /kaggle/input \
        --manifest data/sources/kits23_v0.1.4_files.tsv \
        --out-dir /kaggle/working/spec002 --raw-root /tmp/nnUNet_raw

Outputs in ``--out-dir``: verify_report.csv, case_features.csv, split.json,
splits_final.json. The nnU-Net raw dataset is written under ``--raw-root``.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import shutil
import statistics
import sys
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "src"))

from project3d.data import features, manifest, nnunet_export, split

IMAGE_MIN_BYTES = 10 * 1024 * 1024  # CT volumes are tens of MB; label files are ~1 MB


def detect_roots(input_root: Path, min_files: int = 1,
                 link_dir: Path | None = None, link_mode: str = "symlink") -> tuple[Path, Path]:
    """Find the images folder and the segmentations folder by typical file size.

    Segmentation folders with fewer than ``min_files`` case files are ignored (e.g.
    nnU-Net's validation predictions sitting next to the full dataset). Images may
    be split over several folders (one per Kaggle staging shard, D-015); they are
    then linked into ``link_dir/images`` so the pipeline sees one folder.
    """
    by_dir: dict[Path, list[int]] = {}
    for paths in manifest.find_case_files(input_root).values():
        for p in paths:
            by_dir.setdefault(p.parent, []).append(p.stat().st_size)
    if not by_dir:
        raise SystemExit(f"no case_XXXXX.nii(.gz) files found under {input_root}")
    big = sorted(d for d, s in by_dir.items() if statistics.median(s) >= IMAGE_MIN_BYTES)
    small = [d for d, s in by_dir.items()
             if statistics.median(s) < IMAGE_MIN_BYTES and len(s) >= min_files]
    n_images = sum(len(by_dir[d]) for d in big)
    if not big or n_images < min_files or len(small) != 1:
        raise SystemExit(f"could not tell images from segmentations; folders found: "
                         f"{ {str(d): len(s) for d, s in by_dir.items()} }. "
                         f"Pass --images-root and --segs-root explicitly.")
    if len(big) == 1:
        return big[0], small[0]
    if link_dir is None:
        raise SystemExit(f"images are split over {len(big)} folders; pass a link folder")
    merged = Path(link_dir) / "images"
    if merged.exists():
        shutil.rmtree(merged)
    for d in big:
        for p in sorted(d.iterdir()):
            if not manifest.CASE_FILE_RE.match(p.name):
                continue
            if (merged / p.name).exists():
                raise SystemExit(f"{p.name} appears in more than one image folder")
            nnunet_export._place(p, merged / p.name, link_mode)
    print(f"linked {n_images} images from {len(big)} folders into {merged}")
    return merged, small[0]


def write_csv(path: Path, rows: list[dict]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--manifest", type=Path, required=True)
    ap.add_argument("--input-root", type=Path, help="auto-detect images/segmentation folders here")
    ap.add_argument("--images-root", type=Path)
    ap.add_argument("--segs-root", type=Path)
    ap.add_argument("--out-dir", type=Path, required=True)
    ap.add_argument("--raw-root", type=Path, required=True)
    ap.add_argument("--link-mode", choices=("symlink", "copy"), default="symlink")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--subset", action="store_true",
                    help="allow missing cases (local smoke test); never use for the real run")
    ap.add_argument("--skip-hash", action="store_true", help="size check only (faster, weaker)")
    args = ap.parse_args(argv)

    t0 = time.time()
    args.out_dir.mkdir(parents=True, exist_ok=True)
    if args.images_root and args.segs_root:
        images_root, segs_root = args.images_root, args.segs_root
    elif args.input_root:
        images_root, segs_root = detect_roots(args.input_root, link_dir=args.raw_root.parent / "kits23_merged",
                                              link_mode=args.link_mode)
    else:
        ap.error("give --input-root, or both --images-root and --segs-root")
    print(f"images: {images_root}\nsegmentations: {segs_root}")

    # 1. Verify every file against the pinned manifest.
    exp = manifest.load_manifest(args.manifest)
    report = (manifest.verify_kind(exp, "image", images_root, not args.skip_hash)
              + manifest.verify_kind(exp, "segmentation", segs_root, not args.skip_hash))
    write_csv(args.out_dir / "verify_report.csv", report)
    counts = manifest.summarize(report)
    print(f"[1/4] verify: {counts}  ({time.time() - t0:.0f}s)")
    failing = {s: n for s, n in counts.items() if s in manifest.FAILING or s == "UNEXPECTED"}
    if args.subset:
        failing.pop(manifest.MISSING, None)
    if failing:
        print("STOP: verification failed; see verify_report.csv")
        return 1
    if counts.get(manifest.DECOMPRESSED):
        print("WARNING: some files were un-gzipped by the platform; their hashes cannot be checked")

    images = manifest.resolved_paths([r for r in report if r["kind"] == "image"])
    segs = manifest.resolved_paths([r for r in report if r["kind"] == "segmentation"])
    cases = sorted(set(images) & set(segs))
    if not args.subset and len(cases) != len({c for c, _ in exp}):
        print(f"STOP: only {len(cases)} cases have both image and label")
        return 1

    # 2. Image/mask geometry must agree (SPEC 001 5.4), then label features per case.
    bad = {c: why for c in cases if (why := features.geometry_mismatch(images[c], segs[c]))}
    if bad:
        print(f"STOP: image/mask geometry mismatch: {bad}")
        return 1
    rows = [features.case_features(c, segs[c]) for c in cases]
    cuts = features.add_size_bins(rows)
    write_csv(args.out_dir / "case_features.csv", rows)
    print(f"[2/4] features: {len(rows)} cases, tumor-size tertile cuts (ml) = "
          f"{cuts[0]:.2f}, {cuts[1]:.2f}  ({time.time() - t0:.0f}s)")

    # 3. Split once, fingerprint, save.
    roles = split.make_split(rows, seed=args.seed)
    record = {
        "spec": "SPEC 002",
        "dataset": "KiTS23 v0.1.4",
        # Line endings normalized so Windows and Kaggle copies hash the same.
        "manifest_sha256": hashlib.sha256(
            args.manifest.read_bytes().replace(b"\r\n", b"\n")).hexdigest(),
        "seed": args.seed,
        "fractions": {"train": 0.70, "val": 0.15, "test": 0.15},
        "strata": "cohort | size_bin | cyst",
        "size_bin_cuts_ml": list(cuts),
        "fingerprint_sha256": split.fingerprint(roles),
        "summary": split.summary(rows, roles),
        "roles": roles,
    }
    (args.out_dir / "split.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
    (args.out_dir / "splits_final.json").write_text(
        json.dumps(split.nnunet_splits(roles), indent=2), encoding="utf-8")
    print(f"[3/4] split: {record['summary']['total']}  fingerprint {record['fingerprint_sha256'][:16]}...")

    # 4. nnU-Net raw dataset (train+val only in imagesTr).
    ds = nnunet_export.build_raw_dataset(roles, images, segs, args.raw_root, mode=args.link_mode)
    nnunet_export.check_no_test_leak(ds, roles)
    print(f"[4/4] nnU-Net raw dataset: {ds}  ({time.time() - t0:.0f}s total)")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/project23d/scripts/eval/evaluate.py
"""SPEC 004: score predictions on the locked test set.

Example (Kaggle)::

    python scripts/eval/evaluate.py --pred-dir /kaggle/working/spec004/pred_pp \
        --segs-root <KiTS23 segmentations> --split split.json --role test \
        --out-dir /kaggle/working/spec004/eval

Writes per_case.csv and summary.json. Co-primary metrics are the official KiTS23
Dice and surface Dice, averaged over the three HECs per case, then over cases
(D-012). ``--no-official`` is for local smoke tests only: the summary is then
marked ``primary_valid: false``.
"""

from __future__ import annotations

import argparse
import csv
import json
import sys
from multiprocessing import Pool
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "src"))

from project3d.data import features, manifest
from project3d.eval import metrics, stats

CROSSCHECK_TOL = 1e-3  # our Dice vs official Dice; larger means a geometry/orientation bug


def score_case(args: tuple) -> dict:
    case_id, pred_path, ref_path, use_official = args
    import nibabel as nib

    why = features.geometry_mismatch(pred_path, ref_path)
    if why:
        raise ValueError(f"{case_id}: prediction/reference geometry mismatch ({why})")
    ref_img = nib.load(str(ref_path))
    ref = np.asanyarray(ref_img.dataobj).astype(np.uint8)
    pred = np.asanyarray(nib.load(str(pred_path)).dataobj).astype(np.uint8)
    bad = set(np.unique(pred).tolist()) - features.VALID_LABELS
    if bad:
        raise ValueError(f"{case_id}: prediction has labels {sorted(bad)}")
    spacing = tuple(float(z) for z in ref_img.header.get_zooms()[:3])
    row = {"case_id": case_id, **metrics.case_metrics(pred, ref, spacing)}
    if use_official:
        from project3d.eval import official

        row.update(official.official_case(pred_path, ref_path))
        row["mean_dice"] = float(np.mean([row[f"{h}_dice_official"] for h in metrics.HECS]))
        row["mean_sd"] = float(np.mean([row[f"{h}_sd_official"] for h in metrics.HECS]))
    else:
        row["mean_dice"] = float(np.mean([row[f"{h}_dice"] for h in metrics.HECS]))
        row["mean_sd"] = float("nan")
    return row


def summarize(rows: list[dict], official: bool) -> dict:
    def col(key: str) -> list:
        return [r[key] for r in rows]

    out = {
        "n_cases": len(rows),
        "official_evaluator": official,
        "primary_valid": official,
        "primary": {"mean_dice": stats.describe_with_ci(col("mean_dice")),
                    "mean_surface_dice": stats.describe_with_ci(col("mean_sd"))},
        "regions": {},
    }
    for name in metrics.REGIONS:
        reg = {m: stats.describe(col(f"{name}_{m}")) for m in ("dice", "iou", "precision", "recall")}
        reg["dice"]["ci95"] = stats.bootstrap_ci(col(f"{name}_dice"))
        reg["hd95_mm"] = stats.describe(col(f"{name}_hd95_mm"))
        reg["abs_volume_error_ml"] = stats.describe(
            [abs(r[f"{name}_pred_ml"] - r[f"{name}_ref_ml"]) for r in rows])
        reg["n_ref_empty"] = int(sum(r[f"{name}_ref_ml"] == 0 for r in rows))
        if official and name in metrics.HECS:
            reg["dice_official"] = stats.describe_with_ci(col(f"{name}_dice_official"))
            reg["surface_dice_official"] = stats.describe_with_ci(col(f"{name}_sd_official"))
        out["regions"][name] = reg
    if official:
        out["dice_crosscheck_max_abs_diff"] = max(
            abs(r[f"{h}_dice"] - r[f"{h}_dice_official"]) for r in rows for h in metrics.HECS)
    return out


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--pred-dir", type=Path, required=True)
    ap.add_argument("--segs-root", type=Path, required=True)
    ap.add_argument("--split", type=Path, required=True)
    ap.add_argument("--role", default="test", choices=("test", "val"))
    ap.add_argument("--out-dir", type=Path, required=True)
    ap.add_argument("--np", type=int, default=1, help="worker processes")
    ap.add_argument("--no-official", action="store_true", help="local smoke test only")
    args = ap.parse_args(argv)

    roles = json.loads(args.split.read_text(encoding="utf-8"))["roles"]
    cases = sorted(c for c, r in roles.items() if r == args.role)
    refs = manifest.find_case_files(args.segs_root)
    missing_pred = [c for c in cases if not (args.pred_dir / f"{c}.nii.gz").exists()]
    missing_ref = [c for c in cases if len(refs.get(c, [])) != 1]
    if missing_pred or missing_ref:
        print(f"STOP: missing predictions {missing_pred}; missing/duplicate references {missing_ref}")
        return 1
    official = not args.no_official
    if official:
        from project3d.eval import official as off

        off.check_config()
    jobs = [(c, args.pred_dir / f"{c}.nii.gz", refs[c][0], official) for c in cases]
    if args.np > 1:
        with Pool(args.np) as pool:
            rows = pool.map(score_case, jobs)
    else:
        rows = [score_case(j) for j in jobs]

    args.out_dir.mkdir(parents=True, exist_ok=True)
    with open(args.out_dir / "per_case.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    summary = {"spec": "SPEC 004", "role": args.role, "pred_dir": str(args.pred_dir),
               "split_fingerprint_sha256": json.loads(args.split.read_text())["fingerprint_sha256"],
               **summarize(rows, official)}
    if official:
        summary["kits23_evaluator"] = off.official_version()
    (args.out_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    p = summary["primary"]
    print(f"{args.role}: n={len(rows)}  mean Dice {p['mean_dice'].get('mean', float('nan')):.4f} "
          f"CI {p['mean_dice']['ci95']}  mean surface Dice "
          f"{p['mean_surface_dice'].get('mean', float('nan')):.4f}")
    if official and summary["dice_crosscheck_max_abs_diff"] > CROSSCHECK_TOL:
        print(f"STOP: our Dice differs from the official Dice by "
              f"{summary['dice_crosscheck_max_abs_diff']:.2e}; check orientation/geometry")
        return 1
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/project23d/scripts/inference/benchmark.py
"""SPEC 008: measure speed and quality of the inference settings, on validation cases only.

Each setting runs in its own process, so its peak memory is measured on its own. The test
set is never touched: the case list comes from the split's validation role (D-014).

Example (Kaggle)::

    python scripts/inference/benchmark.py --model-dir model_package --images-root <images> \
        --segs-root <segmentations> --split split.json --training-validation fold_0/validation \
        --out reports/release/inference_benchmark.json
"""

from __future__ import annotations

import argparse
import json
import statistics
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[2]
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "scripts" / "eval")]

from project3d.data import manifest
from project3d.eval.postprocess import keep_largest_regions
from project3d.inference.predictor import LABELS, NnUNetModel, Options, Predictor

SETTINGS = {
    "full": Options(),
    "no_tta": Options(tta=False),
    "no_tta_step_0.75": Options(tta=False, tile_step=0.75),
}
N_CASES = 6
MATCH_DICE = 0.99  # SPEC 008: default settings must reproduce the training validation result


def choose_cases(roles: dict[str, str], n: int = N_CASES) -> list[str]:
    """Evenly spaced validation cases, so the choice can't favour easy ones."""
    val = sorted(c for c, r in roles.items() if r == "val")
    return [val[i] for i in np.linspace(0, len(val) - 1, min(n, len(val))).astype(int)]


def label_dice(a: np.ndarray, b: np.ndarray) -> float:
    """Mean Dice over the labels present in either volume; 1.0 if both are empty."""
    scores = []
    for label in LABELS.values():
        x, y = a == label, b == label
        if x.any() or y.any():
            scores.append(2 * float((x & y).sum()) / float(x.sum() + y.sum()))
    return float(np.mean(scores)) if scores else 1.0


def summarize(name: str, options: Options, rows: list[dict], peak_gpu_gb: float, peak_cpu_gb: float) -> dict:
    seconds = [r["seconds"] for r in rows]
    return {"setting": name, "tta": options.tta, "tile_step": options.tile_step, "n_cases": len(rows),
            "seconds_per_case_mean": round(statistics.mean(seconds), 1),
            "seconds_per_case_max": round(max(seconds), 1),
            "mean_dice": statistics.mean(r["mean_dice"] for r in rows),
            "mean_surface_dice": statistics.mean(r["mean_sd"] for r in rows),
            "peak_gpu_gb": round(peak_gpu_gb, 2), "peak_cpu_gb": round(peak_cpu_gb, 2), "per_case": rows}


def run_setting(name: str, args: argparse.Namespace, cases: list[str]) -> dict:
    import resource

    import evaluate
    import nibabel as nib
    import torch

    from project3d.eval import official

    official.check_config()
    options = SETTINGS[name]
    images, segs = manifest.find_case_files(args.images_root), manifest.find_case_files(args.segs_root)
    predictor = Predictor(NnUNetModel(args.model_dir, options), options)
    torch.cuda.reset_peak_memory_stats()
    rows = []
    with tempfile.TemporaryDirectory() as tmp:
        for case in cases:
            dst = Path(tmp) / f"{case}.nii.gz"
            start = time.time()
            summary = predictor.predict_file(images[case][0], dst)
            row = {"case_id": case, "seconds": round(time.time() - start, 1), "voxel_count": summary["voxel_count"]}
            scored = evaluate.score_case((case, dst, segs[case][0], True))
            row.update(mean_dice=scored["mean_dice"], mean_sd=scored["mean_sd"])
            if name == "full" and args.training_validation:
                done = nib.load(str(args.training_validation / f"{case}.nii.gz"))
                mine = nib.load(str(dst))
                row["dice_vs_training_validation"] = label_dice(
                    keep_largest_regions(np.asanyarray(done.dataobj).astype(np.uint8)),
                    np.asanyarray(mine.dataobj).astype(np.uint8))
            rows.append(row)
            print(name, json.dumps(row), flush=True)
    return summarize(name, options, rows, torch.cuda.max_memory_allocated() / 1e9,
                     resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6)


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--model-dir", type=Path, required=True)
    ap.add_argument("--images-root", type=Path, required=True)
    ap.add_argument("--segs-root", type=Path, required=True)
    ap.add_argument("--split", type=Path, required=True)
    ap.add_argument("--training-validation", type=Path, help="SPEC 003 fold_0/validation folder")
    ap.add_argument("--n-cases", type=int, default=N_CASES)
    ap.add_argument("--out", type=Path, required=True)
    ap.add_argument("--only", choices=list(SETTINGS), help=argparse.SUPPRESS)  # one setting, one process
    args = ap.parse_args(argv)

    split = json.loads(args.split.read_text(encoding="utf-8"))
    cases = choose_cases(split["roles"], args.n_cases)
    if args.only:
        args.out.write_text(json.dumps(run_setting(args.only, args, cases)), encoding="utf-8")
        return 0

    results = []
    with tempfile.TemporaryDirectory() as tmp:
        for name in SETTINGS:
            part = Path(tmp) / f"{name}.json"
            cmd = [sys.executable, __file__, *(sys.argv[1:] if argv is None else argv), "--only", name]
            cmd[cmd.index("--out") + 1] = str(part)
            subprocess.run(cmd, check=True)
            results.append(json.loads(part.read_text(encoding="utf-8")))

    matches = [r["dice_vs_training_validation"] for r in results[0]["per_case"]
               if "dice_vs_training_validation" in r]
    record = {"spec": "SPEC 008", "role": "val", "cases": cases,
              "split_fingerprint_sha256": split["fingerprint_sha256"],
              "settings": results, "min_dice_vs_training_validation": min(matches) if matches else None}
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(record, indent=2), encoding="utf-8")
    for r in results:
        print(f"{r['setting']}: {r['seconds_per_case_mean']} s per case, mean Dice {r['mean_dice']:.4f}, "
              f"surface Dice {r['mean_surface_dice']:.4f}, GPU {r['peak_gpu_gb']} GB, CPU {r['peak_cpu_gb']} GB")
    if matches and min(matches) < MATCH_DICE:
        print(f"STOP: default settings differ from the training validation result ({min(matches):.4f})")
        return 1
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/project23d/scripts/release/export_model.py
"""SPEC 008: package the trained model for release, with a checksum for every file.

Example (Kaggle)::

    python scripts/release/export_model.py \
        --trainer-dir .../nnUNetTrainer_100epochs_kaggle__nnUNetPlans__3d_fullres \
        --out-dir /kaggle/working/model_package

The folder holds everything nnU-Net needs to load the model, plus the trainer class, the
post-processing rule and the model card. It is uploaded to Hugging Face as it is.
"""

from __future__ import annotations

import argparse
import hashlib
import json
import re
import shutil
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT / "src"))

from project3d.eval.postprocess import N_KIDNEYS
from project3d.inference.predictor import CHECKPOINT, TRAINER_FILE

CHECKSUMS = "checksums.json"
MODEL_CARD = ROOT / "reports/release/model_card.md"
TRAINER_SOURCE = ROOT / "src/project3d/training/nnunet_trainer.py"
FRONT_MATTER = """---
license: cc-by-nc-sa-4.0
library_name: nnunet
pipeline_tag: image-segmentation
tags:
  - medical-imaging
  - ct
  - kidney
  - segmentation
  - kits23
---

"""


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def model_card_text(card: Path = MODEL_CARD) -> str:
    """The model card for Hugging Face. Links into this repository would be dead there."""
    body = card.read_text(encoding="utf-8").replace("\r\n", "\n")
    return FRONT_MATTER + re.sub(r"\[([^\]]+)\]\((?:\.\./|docs/|reports/)[^)]*\)", r"\1", body)


def export(trainer_dir: Path, out_dir: Path) -> dict:
    sources = {"plans.json": trainer_dir / "plans.json", "dataset.json": trainer_dir / "dataset.json",
               f"fold_0/{CHECKPOINT}": trainer_dir / "fold_0" / CHECKPOINT, TRAINER_FILE: TRAINER_SOURCE}
    missing = [str(p) for p in sources.values() if not p.exists()]
    if missing:
        raise FileNotFoundError(f"missing {missing}")
    (out_dir / "fold_0").mkdir(parents=True, exist_ok=True)
    for name, src in sources.items():
        shutil.copy(src, out_dir / name)
    rule = {"rule": "keep the largest connected foreground regions", "n_regions": N_KIDNEYS,
            "connectivity": 26, "decision": "D-021"}
    (out_dir / "postprocessing.json").write_text(json.dumps(rule, indent=2), encoding="utf-8")
    (out_dir / "README.md").write_text(model_card_text(), encoding="utf-8", newline="\n")
    files = sorted(p for p in out_dir.rglob("*") if p.is_file() and p.name != CHECKSUMS)
    record = {p.relative_to(out_dir).as_posix(): {"sha256": sha256(p), "bytes": p.stat().st_size} for p in files}
    (out_dir / CHECKSUMS).write_text(json.dumps(record, indent=2), encoding="utf-8")
    return record


def verify(out_dir: Path) -> list[str]:
    """Names of the files that are missing or don't match their recorded checksum."""
    record = json.loads((out_dir / CHECKSUMS).read_text(encoding="utf-8"))
    return [name for name, entry in record.items()
            if not (out_dir / name).exists() or sha256(out_dir / name) != entry["sha256"]]


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    ap.add_argument("--trainer-dir", type=Path, required=True, help="folder holding plans.json and fold_0/")
    ap.add_argument("--out-dir", type=Path, required=True)
    args = ap.parse_args(argv)
    try:
        record = export(args.trainer_dir, args.out_dir)
    except FileNotFoundError as exc:
        print(f"STOP: {exc}")
        return 1
    print(f"packaged {len(record)} files into {args.out_dir}")
    print(json.dumps(record, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/project23d/reports/release/model_card.md
# Model card — nnU-Net v2, KiTS23, fold 0

**Status:** Evaluated once on the 73 internal test cases. Mean Dice is 0.722 (95% CI
0.665–0.777) and mean surface Dice is 0.596 (0.550–0.641). Full numbers are in
[`reports/evaluation/results.md`](../evaluation/results.md).

> Research demo, not for clinical use.

## What it is

- **Task:** 3D semantic segmentation of contrast-enhanced abdominal CT.
- **Outputs:** voxel labels 0 background, 1 kidney, 2 tumor, 3 cyst. The model is trained on the KiTS regions: kidney + masses, masses, tumor.
- **Method:** the official nnU-Net v2 (2.8.1), `3d_fullres`, `nnUNetTrainer_100epochs_kaggle` (the standard 100-epoch recipe, saving checkpoints more often, D-016). One fold only (fold 0), with post-processing chosen on validation that keeps the two largest foreground regions (D-012, D-021).
- **Data:** KiTS23 v0.1.4, 489 labeled cases, split at case level into 343 train, 73 validation and 73 locked test cases (seed 42). Two oversized whole-body scans were left out of training, so 341 cases were trained on (D-017). The split is stratified by cohort proxy, tumor size and cyst presence.

## Intended use

This is a portfolio and research project, built to learn and to show a correctly measured 3D medical segmentation workflow. **It is not a medical device.**

- It must not be used for diagnosis, treatment planning or any decision about a patient.
- It has no regulatory clearance and no clinical validation.

## How it was evaluated

The model was run once on the 73 internal test cases. The co-primary metrics are the official KiTS23 Dice and surface Dice, averaged over the three regions per case and then over cases, each with a bootstrap 95% CI.

The report also includes:

- secondary metrics: IoU, precision, recall, HD95 and volume error;
- an analysis by tumor size, cyst presence and cohort (SPEC 005).

## Known limitations

- **Short training:** one fold and 100 epochs instead of nnU-Net's 5 folds × 1000 epochs, so expect lower scores than the published nnU-Net KiTS23 results.
- **False regions:** the raw model marks false regions away from the kidneys in most scans. Post-processing keeps the two largest regions (D-021), which fails when a false region is bigger than a real kidney. On the test set it made 7 of 73 cases slightly worse, and case_00540 scored 0 on every region.
- **Small tumors:** the small-tumor group has a tumor Dice of 0.324, against 0.835 for large tumors.
- **Cysts:** cysts are effectively not separated from tumors. Median cyst Dice is 0.000 across the 37 test cases that have cysts.
- **Internal test set:** the test set comes from the public KiTS23 training cases. It is not the hidden challenge test set and not an external hospital, so the results can't be compared directly with the leaderboard.
- **Small groups:** 73 test cases, so the sub-group results (for example small tumors) are uncertain. They are descriptive only.
- **No contrast-phase data:** the contrast phase per case is unknown (D-010), so phase effects can't be measured.
- **Dataset biases:** biases in patient population, scanners and annotation are inherited from KiTS23.
- **License:** data CC BY-NC-SA 4.0. The model weights are derived from that data, so non-commercial use only.

## Inference settings

Measured on six validation cases on a Kaggle T4 GPU (`inference_benchmark.json`). The test set was
not used. Peak memory was the same for all three: 4.5 GB of GPU memory and 12.7 GB of RAM.

| Setting | Mean time per scan | Slowest scan | Mean Dice | Mean surface Dice |
|---|---|---|---|---|
| Full: test-time augmentation, tile step 0.5 | 442 s | 608 s | 0.566 | 0.434 |
| No test-time augmentation, tile step 0.5 | 133 s | 222 s | 0.566 | 0.427 |
| No test-time augmentation, tile step 0.75 (**live demo**) | 103 s | 182 s | 0.644 | 0.472 |

The full setting matches the evaluation: Dice of at least 0.9995 against the saved validation
predictions. The published scores and the run-it-yourself notebook use it.

The live demo uses the last setting because it is about four times faster. Its higher Dice is not an
improvement. It comes almost entirely from one scan, case_00002, which went from 0.267 to 0.788, and
six cases can't support more than "no loss was measured". The demo setting was not scored on the
73 test cases.

## How to use it

Install `nnunetv2==2.8.1`, then copy `project23d_trainer.py` into nnU-Net's
`training/nnUNetTrainer/variants/` folder. The checkpoint names that class, so nnU-Net can't load
the model without it. Keep the two largest connected foreground regions of the output
(26-connected), as described in `postprocessing.json`. The published scores depend on that step.

`checksums.json` holds the SHA-256 of every file in this package.

## Credits and license

- **Data:** KiTS23 (<https://github.com/neheller/kits23>), CC BY-NC-SA 4.0.
- **Method:** nnU-Net (Isensee et al., *Nature Methods* 2021; <https://github.com/MIC-DKFZ/nnUNet>), Apache 2.0.
- **These weights:** CC BY-NC-SA 4.0, because they are derived from KiTS23. Non-commercial use only, shared under the same terms.

See also [`docs/limitations.md`](../../docs/limitations.md).


In [ ]:
# Step 1 - install nnU-Net (same pinned version as training) and the official evaluator
def sh(cmd):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
sh([sys.executable, "-m", "pip", "install", "-q", "nnunetv2==2.8.1", "nibabel",
    "git+https://github.com/neheller/kits23"])
for var in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):  # silences nnU-Net warnings
    os.environ.setdefault(var, f"/tmp/{var}")

In [ ]:
# Step 2 - find the inputs: KiTS23 folders, trained model, locked split
sys.path[:0] = [f"{WORK}/src", f"{WORK}/scripts/data"]
import prepare_kits23
INPUT = Path("/kaggle/input")
IMAGES, SEGS = prepare_kits23.detect_roots(INPUT, min_files=489, link_dir=Path("/tmp/kits23_merged"))
ckpts = sorted(INPUT.rglob("fold_0/checkpoint_final.pth"))
assert len(ckpts) == 1, f"need exactly one fold_0/checkpoint_final.pth, found {ckpts}"
FOLD = ckpts[0].parent
splits = [p for p in INPUT.rglob("split.json") if "roles" in json.loads(p.read_text())]
assert len(splits) == 1, f"need exactly one SPEC 002 split.json, found {splits}"
OUT = Path("/kaggle/working/spec008")
OUT.mkdir(parents=True, exist_ok=True)
PACKAGE = Path("/kaggle/working/model_package")

In [ ]:
# Step 3 - package the model and verify the checksums
sh([sys.executable, f"{WORK}/scripts/release/export_model.py", "--trainer-dir", FOLD.parent, "--out-dir", PACKAGE])
sys.path.insert(0, f"{WORK}/scripts/release")
import export_model
assert export_model.verify(PACKAGE) == [], "checksum mismatch"
shutil.copy(PACKAGE / "checksums.json", OUT / "model_checksums.json")

In [ ]:
# Step 4 - three settings on validation cases. The default setting is also compared with
# the prediction saved in SPEC 003's validation, which proves this predictor matches evaluation.
sh([sys.executable, f"{WORK}/scripts/inference/benchmark.py", "--model-dir", PACKAGE, "--images-root", IMAGES,
    "--segs-root", SEGS, "--split", splits[0], "--training-validation", FOLD / "validation",
    "--out", OUT / "inference_benchmark.json"])
print(f"DONE in {(time.time() - T0) / 3600:.2f} h")